In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

def compute_did_with_pretrend(df: pd.DataFrame) -> dict:
    # ── Input validation ──────────────────────────────────
    assert 'week'    in df.columns, "missing 'week' column"
    assert 'group'   in df.columns, "missing 'group' column"
    assert 'minutes' in df.columns, "missing 'minutes' column"

    # ── Step 1: Copy to avoid mutating caller's df ────────
    df = df.copy()

    # ── Step 2: Create pre/post flag ──────────────────────
    # week < 0 → pre (0), week > 0 → post (1)
    df['on_off'] = (df['week'] > 0).astype(int)

    # ── Step 3: Compute group × period means ──────────────
    means = df.groupby(['group', 'on_off'])['minutes'].mean()

    treat_pre    = means.loc[('treat',   0)]
    treat_post   = means.loc[('treat',   1)]
    control_pre  = means.loc[('control', 0)]
    control_post = means.loc[('control', 1)]

    # ── Step 4: DiD formula ───────────────────────────────
    delta_treat   = treat_post   - treat_pre
    delta_control = control_post - control_pre
    did_estimate  = delta_treat  - delta_control

    conclusion = (
        'Positive effect'    if did_estimate > 0 else
        'Negative effect'    if did_estimate < 0 else
        'No effect detected'
    )

    # ── Step 5: Parallel trends test (pre-period only) ────
    pre_df = df[df['week'] < 0].copy()   # .copy() avoids SettingWithCopyWarning
    pre_df['treat']       = (pre_df['group'] == 'treat').astype(int)
    pre_df['interaction'] = pre_df['treat'] * pre_df['week']

    X     = sm.add_constant(pre_df[['treat', 'week', 'interaction']])
    model = sm.OLS(pre_df['minutes'], X).fit()

    pretrend_pval     = model.pvalues['interaction']
    pretrend_violated = bool(pretrend_pval < 0.05)   # cast to Python bool

    # ── Output validation ─────────────────────────────────
    assert 0 <= pretrend_pval <= 1,          "pretrend_pval must be in [0, 1]"
    assert isinstance(pretrend_violated, bool)
    assert conclusion in (
        'Positive effect', 'Negative effect', 'No effect detected'
    )

    return {
        'did_estimate'     : did_estimate,
        'treat_pre'        : treat_pre,
        'treat_post'       : treat_post,
        'control_pre'      : control_pre,
        'control_post'     : control_post,
        'pretrend_pval'    : pretrend_pval,
        'pretrend_violated': pretrend_violated,
        'conclusion'       : conclusion
    }




In [3]:
# ── Test ──────────────────────────────────────────────────
np.random.seed(0)
rows = []
for uid in range(200):
    group = 'treat' if uid < 100 else 'control'
    for week in [-2, -1, 1, 2]:
        base      = 30 if group == 'treat' else 25
        treatment = 5  if (group == 'treat' and week > 0) else 0
        rows.append({
            'user_id': uid, 'week': week,
            'group': group,
            'minutes': base + treatment + np.random.normal(0, 3)
        })

df     = pd.DataFrame(rows)
result = compute_did_with_pretrend(df)

assert abs(result['did_estimate'] - 5) < 2
assert 0 <= result['pretrend_pval'] <= 1
assert isinstance(result['pretrend_violated'], bool)
print("All tests passed ✅")

for k, v in result.items():
    print(f"{k:20s}: {v}")

All tests passed ✅
did_estimate        : 4.4929390776761515
treat_pre           : 29.867838930961867
treat_post          : 34.960849097803255
control_pre         : 24.344431373356027
control_post        : 24.944502462521264
pretrend_pval       : 0.2997308737253748
pretrend_violated   : False
conclusion          : Positive effect
